# CNN Theory:
CNNs are specialized neural networks designed for processing grid-like data such as images
# Key components:
1. Convolutional Layer: applies filters/kernels to detect features (edges, textures, patterns)
2. Pooling Layer: reduces spatial dimensions while retaining important information
3. Activation Function: introduces non-linearity (typically ReLU)
4. Fully Connected Layer: final classification/regression layer

# How convolution works:
- A filter (small matrix) slides across the input image
- At each position, element-wise multiplication and summation occurs
- This creates a feature map highlighting specific patterns
- Multiple filters detect different features (horizontal edges, vertical edges, etc.)

#Pooling reduces overfitting and computational cost:
- Max pooling: takes maximum value in each region
- Average pooling: takes average value in each region

CNN Architecture typically follows: Input -> Conv -> ReLU -> Pool -> Conv -> ReLU -> Pool -> FC -> Output
Production-ready CNN model example (using PyTorch-like pseudocode):

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class ProductionCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(ProductionCNN, self).__init__()
        
        # First convolutional block
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)  # 3 input channels (RGB), 64 output channels
        self.bn1 = nn.BatchNorm2d(64)  # Batch normalization for stable training
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2)  # Reduce spatial dimensions by half
        self.dropout1 = nn.Dropout2d(0.25)  # Prevent overfitting
        
        # Second convolutional block
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.dropout2 = nn.Dropout2d(0.25)
        
        # Third convolutional block
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.dropout3 = nn.Dropout2d(0.25)
        
        # Global Average Pooling (alternative to flattening)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully connected layers
        self.fc1 = nn.Linear(256, 512)
        self.dropout4 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)
        
    def forward(self, x):
        # First block
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.dropout1(x)
        
        # Second block
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.dropout2(x)
        
        # Third block
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.relu(self.bn6(self.conv6(x)))
        x = self.pool3(x)
        x = self.dropout3(x)
        
        # Global average pooling and classification
        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout4(x)
        x = self.fc2(x)
        
        return x

# Training configuration for production
def train_production_cnn():
    model = ProductionCNN(num_classes=10)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)
    
    # Data augmentation for better generalization
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))  # ImageNet stats
    ])
    
    # Training loop with validation
    for epoch in range(100):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        
        scheduler.step()
        
        # Validation phase
        model.eval()
        with torch.no_grad():
            val_loss = 0
            correct = 0
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(output, target).item()
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()
